# Exploratory Data Analysis: Spectral Feature Distributions

Before training any model, it helps to actually look at the data. This notebook digs into the audio files to understand:

- How class-balanced is the dataset?
- Do different genres have visually distinct mel spectrograms?
- Are there any outlier files (very short, very quiet, corrupted)?
- What do the raw waveforms look like across genres?

This is not training code. It is purely investigative.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torchaudio
import torchaudio.transforms as T
import torch
from pathlib import Path
from collections import defaultdict

random.seed(42)
np.random.seed(42)

DATA_ROOT = Path('/teamspace/studios/this_studio/data/messy_mashup')
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop',
          'jazz', 'metal', 'pop', 'reggae', 'rock']
SR = 22050

## 1. Class Distribution

First, count how many audio files exist per genre. A heavily imbalanced dataset would bias the model toward majority classes.

In [ ]:
counts = {}
for genre in GENRES:
    genre_dir = DATA_ROOT / 'train' / genre
    if genre_dir.exists():
        counts[genre] = len(list(genre_dir.glob('*.wav')))
    else:
        counts[genre] = 0

df_counts = pd.DataFrame(list(counts.items()), columns=['genre', 'count'])
df_counts = df_counts.sort_values('count', ascending=False).reset_index(drop=True)
print(df_counts)
print(f"\nTotal files: {df_counts['count'].sum()}")
print(f"Min: {df_counts['count'].min()} | Max: {df_counts['count'].max()}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
colors = plt.cm.tab10(np.linspace(0, 1, len(GENRES)))
bars = ax.bar(df_counts['genre'], df_counts['count'], color=colors)
ax.set_xlabel('Genre')
ax.set_ylabel('Number of files')
ax.set_title('Training samples per genre')
for bar, val in zip(bars, df_counts['count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

## 2. Audio Duration Distribution

Models expect fixed-length input. Files that are shorter than expected will need padding; files that are longer get cropped. Let us check whether most files are close to the same duration.

In [ ]:
duration_data = defaultdict(list)

for genre in GENRES:
    genre_dir = DATA_ROOT / 'train' / genre
    if not genre_dir.exists():
        continue
    files = list(genre_dir.glob('*.wav'))
    sample = random.sample(files, min(30, len(files)))
    for f in sample:
        try:
            info = torchaudio.info(str(f))
            duration = info.num_frames / info.sample_rate
            duration_data['genre'].append(genre)
            duration_data['duration_sec'].append(duration)
            duration_data['sample_rate'].append(info.sample_rate)
        except Exception as e:
            print(f"Could not read {f.name}: {e}")

df_dur = pd.DataFrame(duration_data)
print(df_dur.groupby('genre')['duration_sec'].describe().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
genres_sorted = df_dur.groupby('genre')['duration_sec'].median().sort_values().index.tolist()
bp_data = [df_dur[df_dur['genre'] == g]['duration_sec'].values for g in genres_sorted]
ax.boxplot(bp_data, labels=genres_sorted, patch_artist=True)
ax.set_xlabel('Genre')
ax.set_ylabel('Duration (seconds)')
ax.set_title('Audio duration distribution by genre')
ax.axhline(y=30, color='red', linestyle='--', alpha=0.5, label='30s mark')
ax.legend()
plt.tight_layout()
plt.savefig('duration_distribution.png', dpi=150)
plt.show()

## 3. Waveform Comparison Across Genres

Raw amplitude over time tells us about energy patterns. Metal tends to be uniformly loud (clipped waveforms). Classical has dynamic range. Hip-hop has punchy transients from drums and kicks.

In [ ]:
def load_audio_mono(path, target_sr=SR, duration_sec=5.0):
    waveform, sr = torchaudio.load(str(path))
    if sr != target_sr:
        waveform = T.Resample(sr, target_sr)(waveform)
    waveform = waveform.mean(dim=0)  # stereo to mono
    n_samples = int(target_sr * duration_sec)
    if waveform.shape[0] >= n_samples:
        start = (waveform.shape[0] - n_samples) // 2
        waveform = waveform[start:start + n_samples]
    else:
        waveform = torch.nn.functional.pad(waveform, (0, n_samples - waveform.shape[0]))
    return waveform.numpy()

fig, axes = plt.subplots(5, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, genre in enumerate(GENRES):
    genre_dir = DATA_ROOT / 'train' / genre
    files = list(genre_dir.glob('*.wav')) if genre_dir.exists() else []
    if not files:
        axes[idx].set_title(f'{genre} (no files)')
        continue
    f = random.choice(files)
    waveform = load_audio_mono(f)
    t = np.linspace(0, 5, len(waveform))
    axes[idx].plot(t, waveform, linewidth=0.4, color='steelblue')
    axes[idx].set_title(genre.capitalize(), fontsize=11, fontweight='bold')
    axes[idx].set_ylim(-1.1, 1.1)
    axes[idx].set_xlabel('Time (s)')
    axes[idx].set_ylabel('Amplitude')

plt.suptitle('Waveform comparison (5-second centre crop)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('waveform_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Mel Spectrogram Comparison

The mel spectrogram is the primary input for our CNN models. Different genres should show distinct texture patterns:
- Classical: sparse, harmonic vertical lines
- Metal: dense broadband noise across the whole frequency range
- Hip-hop: strong low-frequency bass energy
- Reggae: rhythmic off-beat patterns

In [ ]:
mel_transform = T.MelSpectrogram(
    sample_rate=SR,
    n_fft=1024,
    hop_length=512,
    n_mels=128,
    f_min=20,
    f_max=8000
)
db_transform = T.AmplitudeToDB(top_db=80)

fig, axes = plt.subplots(5, 2, figsize=(14, 15))
axes = axes.flatten()

for idx, genre in enumerate(GENRES):
    genre_dir = DATA_ROOT / 'train' / genre
    files = list(genre_dir.glob('*.wav')) if genre_dir.exists() else []
    if not files:
        axes[idx].set_title(f'{genre} (no files)')
        continue
    f = random.choice(files)
    waveform = load_audio_mono(f)
    waveform_t = torch.from_numpy(waveform).unsqueeze(0)
    mel = mel_transform(waveform_t)
    mel_db = db_transform(mel).squeeze(0).numpy()

    im = axes[idx].imshow(
        mel_db, aspect='auto', origin='lower',
        cmap='magma', vmin=-80, vmax=0
    )
    axes[idx].set_title(genre.capitalize(), fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Time frames')
    axes[idx].set_ylabel('Mel bins')
    plt.colorbar(im, ax=axes[idx], format='%+2.0f dB')

plt.suptitle('Mel spectrogram comparison (128 mel bins, centre crop)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('mel_spectrogram_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. RMS Energy Distribution

RMS (Root Mean Square) energy is a measure of loudness. Genres like metal and disco are expected to have consistently high energy, while classical has high variance (quiet passages vs loud climaxes).

In [ ]:
rms_data = defaultdict(list)

for genre in GENRES:
    genre_dir = DATA_ROOT / 'train' / genre
    if not genre_dir.exists():
        continue
    files = list(genre_dir.glob('*.wav'))
    sample = random.sample(files, min(40, len(files)))
    for f in sample:
        try:
            waveform = load_audio_mono(f)
            rms = np.sqrt(np.mean(waveform ** 2))
            rms_data['genre'].append(genre)
            rms_data['rms'].append(rms)
        except Exception:
            pass

df_rms = pd.DataFrame(rms_data)
genre_order = df_rms.groupby('genre')['rms'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(11, 4))
bp_data = [df_rms[df_rms['genre'] == g]['rms'].values for g in genre_order]
ax.boxplot(bp_data, labels=genre_order, patch_artist=True)
ax.set_xlabel('Genre')
ax.set_ylabel('RMS Energy')
ax.set_title('RMS energy distribution by genre (higher = louder on average)')
plt.tight_layout()
plt.savefig('rms_energy.png', dpi=150)
plt.show()

## Key Takeaways

After running this EDA, note the following before any training decisions:

1. **Class balance**: If any genre has significantly fewer samples, weighted loss (`CrossEntropyLoss(weight=...)`) may help.
2. **Duration**: Most files are close to the expected length, but edge cases exist and need handling in the DataLoader.
3. **Energy**: Classical and jazz have high dynamic range, which is why `AmplitudeToDB` with `top_db=80` is important rather than raw mel power.
4. **Spectral texture**: Metal and classical look very different on a mel spectrogram, which is encouraging. Genre pairs like rock vs metal may be harder to separate.